In [ ]:
import pandas as pd

bottleneck = 100

# Load PCA reconstruction data
pca_path = f"../data/AE_outputs/engcensus_all/PCA/{bottleneck}_components.csv"
pca_reco = pd.read_csv(pca_path, index_col=0)

cleaned_data_path = "../data/census_data/engcensus_cleaned_scaled.parquet"
data = pd.read_parquet(cleaned_data_path)
data.set_index('OA', inplace=True)

abs_diff = abs(data - pca_reco)
abs_diff_mean = abs_diff.mean(axis=1)
norm_diff = abs_diff_mean / abs_diff_mean.mean() * 100

# AE reconstruction
reco_path = f"../data/AE_outputs/engcensus_all/250epoch_scan_lin/census_geodemo__bottleneck_{bottleneck}_v1__reconstructed_outputs.csv"
reco_data = pd.read_csv(reco_path, index_col=0)

# Compute absolute differences and normalized differences for AE
ae_abs_diff = abs(data - reco_data)
#rmse instead of abs
ae_abs_diff_mean = ae_abs_diff.mean(axis=1)
ae_norm_diff = ae_abs_diff_mean / ae_abs_diff_mean.mean() * 100

#add a row for the precent change ae_abs_diff - abs_diff / abs_diff
perc_change = (ae_abs_diff_mean - abs_diff_mean) / abs_diff_mean * 100
# Optional: Combine results for comparison
comparison_df = pd.DataFrame({
    "PCA_norm_diff": norm_diff,
    "AE_norm_diff": ae_norm_diff,
    "perc_change": perc_change
})

In [ ]:
# # Optional: print or plot summary stats
# print(comparison_df.describe())

#sort by PCA norm diff
comparison_df = comparison_df.sort_values(by='PCA_norm_diff', ascending=False)
print(comparison_df.head(5))

#Sort by AE norm diff
comparison_df = comparison_df.sort_values(by='AE_norm_diff', ascending=False)
print(comparison_df.head(5))

#Sort by percent change
comparison_df = comparison_df.sort_values(by='perc_change', ascending=True)
print(comparison_df.head(5))
# Save the comparison DataFrame to a CSV

In [ ]:
oa = 'E00187137'  # Change OA here

print(f"\n--- Reconstruction error analysis for OA {oa} ---")

# Overall normalized error
print(f"PCA normalized error: {norm_diff.loc[oa]:.2f}%")
print(f"AE normalized error: {ae_norm_diff.loc[oa]:.2f}%\n")

# Relative per-variable contribution to error
pca_row = abs_diff.loc[oa] / abs_diff_mean.loc[oa]
ae_row = ae_abs_diff.loc[oa] / ae_abs_diff_mean.loc[oa]

# Round and sort
pca_top = pca_row.round(5).sort_values(ascending=False)
ae_top = ae_row.round(5).sort_values(ascending=False)

# Print top N for PCA
top_n = 10
print(f"Top {top_n} PCA error-contributing variables:")
print(f"{'Rank':<5} {'Variable':<30} {'Contribution':>12}")
print("-" * 50)
for i, (var, val) in enumerate(pca_top.head(top_n).items(), 1):
    print(f"{i:<5} {var:<30} {val:>12.5f}")

print("\n" + "=" * 50 + "\n")

# Print top N for AE
print(f"Top {top_n} AE error-contributing variables:")
print(f"{'Rank':<5} {'Variable':<30} {'Contribution':>12}")
print("-" * 50)
for i, (var, val) in enumerate(ae_top.head(top_n).items(), 1):
    print(f"{i:<5} {var:<30} {val:>12.5f}")
